# Embeddings

This file teaches you how to turn text into numbers (embeddings) so you can search by meaning.
By the end, you will know how to create embeddings, compare them, and use different embedding types.

## Setup

Run this cell first. It loads the libraries and connects to OpenAI.

In [1]:
!pip install python-dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("API_KEY")

In [3]:
import numpy as np
from openai import OpenAI

client = OpenAI(api_key=api_key)

In [4]:
MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

**What happened:** We imported numpy (a math library) and created an OpenAI client. We also set the model names we will use.

### Helper Functions

**Embedding** is an array of numbers that represents the meaning of a piece of text. Texts with similar meanings have similar embeddings.

In [6]:
def get_embedding(text):
    
    """Send one text to OpenAI and get back its embedding."""
    
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=text
    )

    return response.data[0].embedding

In [7]:
def get_embeddings(texts):

    """Send multiple texts at once. Cheaper and faster."""

    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts
    )

    embeddings = []

    for item in response.data:
        embeddings.append(item.embedding)

    return embeddings

In [8]:
def cosine_similarity(a, b):

    """Return a number between -1 and 1.
    1 = identical meaning. 0 = unrelated."""

    a,b = np.array(a), np.array(b)

    return float(
        np.dot(a,b) / (np.linalg.norm(a) * np.linalg.norm(b))
    )

**What happened:** We created three helper functions.
- `get_embedding` turns one text into an embedding.
- `get_embeddings` turns many texts into embeddings in one API call.
- `cosine_similarity` measures how similar two embeddings are.

## Dense Embeddings

**What it does:** Converts text into an array of numbers where every position has a value.

**When to use it:** Whenever you need to search text by meaning instead of exact words.

**Steps:**
1. Send your text to the OpenAI embeddings API.
2. Get back an array of 1536 numbers.
3. Use this array to compare with other texts.

**Dense** means every position in the array has a non-zero value.

In [10]:
text = "Kubernetes uses containers to deploy applications"

embedding = get_embedding(text=text)

print(embedding[:5])
print(len(embedding))

[-0.02197265625, 0.00193023681640625, 0.057525634765625, 0.021087646484375, 0.06396484375]
1536


**What happened:** We turned one sentence into 1536 numbers. These numbers represent the meaning of the sentence.

In [14]:
import json

texts = [
    "How do I reset my password?",
    "I forgot my login credentials",
    "What is the weather today?"
]

embeddings = get_embeddings(texts=texts)

print(len(embeddings))

for embedding in embeddings:
    print(embedding[:5])

3
[0.017608642578125, -0.04571533203125, 0.0298004150390625, 0.0219268798828125, -0.048675537109375]
[0.010955810546875, -0.0028781890869140625, 0.00557708740234375, -0.047027587890625, 0.00839996337890625]
[-0.00878143310546875, -0.00768280029296875, -0.01220703125, -0.0008749961853027344, -0.006961822509765625]


**What happened:** We embedded 3 texts in a single API call. This is cheaper and faster than calling the API 3 times.

## Cosine Similarity

**What it does:** Measures how similar two embeddings are. Returns a number between -1 and 1.

**When to use it:** After you create embeddings, use cosine similarity to find which texts are most related.

**Steps:**
1. Get embeddings for all your texts.
2. Compare each pair using `cosine_similarity`.
3. Higher score = more similar meaning.

In [15]:
texts = [
    "How do I reset my password?",
    "I forgot my login credentials",
    "what is Kubernetes?"
]

embeddings = get_embeddings(texts=texts)

In [17]:
sim_01 = cosine_similarity(embeddings[0], embeddings[1])
sim_02 = cosine_similarity(embeddings[0], embeddings[2])

print(f"{sim_01:.3f}")
print(f"{sim_02:.3f}")

0.576
0.012


**What happened:** The two authentication-related sentences got a high similarity score. The unrelated sentence (Kubernetes) got a low score. This is how embedding-based search works: find the texts with the highest similarity to your query.

## Reduced Dimensions (Matryoshka Embeddings)

**What it does:** Creates shorter embeddings (fewer numbers) to save storage and speed up search.

**When to use it:** When you have millions of texts and need to save memory or search faster.

**Steps:**
1. Pass the `dimensions` parameter to the API.
2. Get a shorter embedding (e.g., 256 instead of 1536).
3. Quality drops slightly but you save storage.

In [18]:
# Full embedding: 1536 dimensions

text = "Kubernetes autoscaling with HPA"

embedding_full = get_embedding(text=text)

print(len(embedding_full))

1536


In [19]:
# Reduced embedding: 256 dimensions

response = client.embeddings.create(
    model=EMBED_MODEL,
    input=text,
    dimensions=256
)

embedding_256 = response.data[0].embedding

print(len(embedding_256))

256


**What happened:** We created two embeddings of the same text. The full version has 1536 numbers (6 KB). The reduced version has 256 numbers (1 KB). The reduced version is 6x smaller but keeps about 95% of the quality.

## Sparse Embeddings (TF-IDF)

**What it does:** Creates embeddings where most values are zero. Only words that appear in the text get non-zero values.

**When to use it:** When you need exact keyword matching. Dense embeddings can miss specific terms like product names or error codes.

**Steps:**
1. Build a vocabulary from all your documents.
2. For each document, score how important each word is.
3. Search by matching words between query and documents.

**TF-IDF** stands for Term Frequency - Inverse Document Frequency. It gives high scores to words that appear often in one document but rarely across all documents.

In [20]:
!pip install scikit-learn

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.22.1-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 70.1 MB/s  0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached narwhals-2.22.1-py3-none-any.whl (454 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 79.2 MB/s  0:00:00 eta 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cos

corpus = [
    "Kubernetes Pod autoscaling with HPA",
    "Docker container runtime configuration",
    "Kubernetes Service types ClusterIP NodePort"
]

In [46]:
tfidf = TfidfVectorizer()

sparse_matrix = tfidf.fit_transform(corpus)

print(tfidf.vocabulary_, "\n")

print(sparse_matrix)

{'kubernetes': 6, 'pod': 8, 'autoscaling': 0, 'with': 12, 'hpa': 5, 'docker': 4, 'container': 3, 'runtime': 9, 'configuration': 2, 'service': 10, 'types': 11, 'clusterip': 1, 'nodeport': 7} 

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 14 stored elements and shape (3, 13)>
  Coords	Values
  (0, 6)	0.3554324678504174
  (0, 8)	0.46735098181071627
  (0, 0)	0.46735098181071627
  (0, 12)	0.46735098181071627
  (0, 5)	0.46735098181071627
  (1, 4)	0.49999999999999994
  (1, 3)	0.49999999999999994
  (1, 9)	0.49999999999999994
  (1, 2)	0.49999999999999994
  (2, 6)	0.3554324678504174
  (2, 10)	0.46735098181071627
  (2, 11)	0.46735098181071627
  (2, 1)	0.46735098181071627
  (2, 7)	0.46735098181071627


In [47]:
query = "Kubernetes autoscaling pods"

query_sparse = tfidf.transform([query])

In [48]:
scores = sk_cos(query_sparse, sparse_matrix)[0]

print(scores)

[0.58715345 0.         0.21516051]


In [43]:
# Sort indices by score (highest first)

sorted_indices = scores.argsort()[::-1]
print(sorted_indices, "\n")


for idx in sorted_indices:
    score = scores[idx]
    print(f"{score:.3f} - {corpus[idx]}")

[0 2 1] 

0.587 - Kubernetes Pod autoscaling with HPA
0.215 - Kubernetes Service types ClusterIP NodePort
0.000 - Docker container runtime configuration


**What happened:** TF-IDF found the document about Pod autoscaling because it shares exact keywords with the query. Sparse search is good at matching specific terms that dense search might miss.

## Hybrid Search (Dense + Sparse)

**What it does:** Combines dense (meaning-based) and sparse (keyword-based) search for better results.

**When to use it:** In production systems where you need both meaning-based and keyword-based matching.

**Steps:**
1. Run dense search and get a ranked list.
2. Run sparse search and get a ranked list.
3. Combine both lists using Reciprocal Rank Fusion (RRF).

**Reciprocal Rank Fusion (RRF)** is a formula that combines two ranked lists into one. A document that ranks high in both lists gets the highest combined score.

In [49]:
documents = [
    "Kubernetes HPA scales Pods based on CPU metrics",
    "Docker Compose manages multi-container apps",
    "The HPA controller checks metrics every 15 seconds",
    "Container images are stored in registries",
    "Horizontal Pod Autoscaler supports custom metrics",
]

In [51]:
def hybrid_search(query, docs, alpha = 0.5):

    # Dense scores
    query_embedding = get_embedding(query)

    doc_embeddings = get_embeddings(docs)


    dense_scores = []
    for doc_emb in doc_embeddings:
        score = cosine_similarity(query_embedding, doc_emb)
        dense_scores.append(score)

    
    # Sparse scores 
    vectorizer = TfidfVectorizer()
    sparse_matrix = vectorizer.fit_transform(docs)
    query_sparse = vectorizer.transform([query])
    sparse_scores = sk_cos(query_sparse, sparse_matrix)[0]
    sparse_scores = sparse_scores.tolist()


    return dense_scores, sparse_scores

In [54]:
query = "HPA autoscaling metrics"

dense_scores, sparse_scores = hybrid_search(
    query=query, docs=documents
)


for i, doc in enumerate(documents):
    dense = dense_scores[i]
    sparse = sparse_scores[i]

    print(f"{dense:.3f} - {sparse:.3f}")



0.641 - 0.394
0.234 - 0.000
0.628 - 0.394
0.199 - 0.000
0.654 - 0.183


**What happened:** Each document got two scores: one from dense search (meaning) and one from sparse search (keywords). Documents about HPA and metrics scored high on both. In production, you combine these scores using RRF to get the best of both approaches.

## HyDE (Hypothetical Document Embedding)

**What it does:** Instead of embedding the short query, it asks the LLM to write a hypothetical answer, then embeds that answer.

**When to use it:** When queries are short (like "What is HPA?") and documents are long. The hypothetical answer is closer in style to the documents, so it matches better.

**Steps:**
1. Ask the LLM to write a fake answer to the query.
2. Create an embedding of the fake answer.
3. Search using that embedding instead of the query embedding.

In [55]:
def hyde_search(query):
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=100,
        temperature=0,
        messages=[{
            "role": "user",
            "content": "Write a short paragraph "
            + f"answering: {query}"
        }]
    )

    answer = response.choices[0].message.content
    print(answer)
    return answer

In [62]:
query = "How does Kubernetes autoscale?"

hypo = hyde_search(query=query)


query_embedding = get_embedding(query)
hypo_embedding = get_embedding(hypo)
doc_embedddings = get_embeddings(documents)

Kubernetes autoscaling is achieved through two primary mechanisms: the Horizontal Pod Autoscaler (HPA) and the Cluster Autoscaler. The HPA automatically adjusts the number of pod replicas in a deployment based on observed metrics, such as CPU utilization or custom metrics, ensuring that applications can handle varying loads efficiently. It continuously monitors these metrics and scales the number of pods up or down as needed. On the other hand, the Cluster Autoscaler manages the scaling of the underlying infrastructure by adding or removing nodes in


In [64]:
for i, doc in enumerate(documents):
    regular_score = cosine_similarity(query_embedding, doc_embedddings[i])
    hyde_score = cosine_similarity(hypo_embedding, doc_embedddings[i])
    print(f"{regular_score:.3f} - {hyde_score:.3f}")

0.607 - 0.641
0.350 - 0.297
0.268 - 0.334
0.236 - 0.181
0.537 - 0.569


**What happened:** The HyDE approach got higher similarity scores for the relevant documents. The hypothetical answer looks like a document, so it matches real documents better than a short query does.

## Summary

- **Dense embeddings** turn text into arrays of numbers. Similar meanings produce similar arrays.
- **Cosine similarity** measures how similar two embeddings are (1 = identical, 0 = unrelated).
- **Reduced dimensions** make embeddings smaller to save storage. Quality drops slightly.
- **Sparse embeddings (TF-IDF)** match exact keywords. Most values are zero.
- **Hybrid search** combines dense and sparse for the best results.
- **HyDE** generates a fake answer and embeds that instead of the query. Works better for short queries.